Phase 2 — Encoding cleaned ASD data into a numeric feature matrix
suitable for (a) k-NN similarity graph construction in Phase 3, and
(b) node features for the GNN in Phase 5.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
CLEAN_PATH = "Data/data_clean.csv"
OUT_X_PATH = "Data/X_features.csv"
OUT_Y_PATH = "Data/y_labels.csv"

In [3]:
df = pd.read_csv(CLEAN_PATH)
print(f"Loaded cleaned data: {df.shape}")

Loaded cleaned data: (704, 18)



---------------------------------------------------------------
Step 1: Separate target from features immediately.
This must NEVER re-enter the feature matrix -- keeping it isolated
here prevents any accidental leakage later.
---------------------------------------------------------------

In [4]:
y = df["Class/ASD"].copy()
df_features = df.drop(columns=["Class/ASD"])

---------------------------------------------------------------
Step 2: Define column groups by type.
This grouping is the single most important decision in this phase --
it determines how each column contributes to Euclidean distance
in the k-NN graph built in Phase 3.
---------------------------------------------------------------

In [5]:
binary_cols = [f"A{i}_Score" for i in range(1, 11)] + [
    "gender", "jaundice", "family_autism_history"
]
continuous_cols = ["age"]
categorical_cols = ["ethnicity", "relation", "country_grouped"]

assert set(binary_cols + continuous_cols + categorical_cols) == set(df_features.columns), \
    "Column grouping doesn't match dataframe columns -- check for typos/missed columns"

---------------------------------------------------------------
Step 3: One-hot encode the categorical columns.
drop_first=False is deliberate: for similarity/distance computation
we want each category to have its own explicit dimension (unlike
regression, where dropping one avoids multicollinearity -- that
concern doesn't apply here since we're not fitting a linear model
on these columns directly).
---------------------------------------------------------------

In [6]:
df_categorical_encoded = pd.get_dummies(
    df_features[categorical_cols], columns=categorical_cols, drop_first=False
)
print(f"\nCategorical columns expanded from {len(categorical_cols)} "
      f"to {df_categorical_encoded.shape[1]} one-hot columns")
print(f"One-hot columns: {list(df_categorical_encoded.columns)}")



Categorical columns expanded from 3 to 28 one-hot columns
One-hot columns: ['ethnicity_Asian', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Latino', 'ethnicity_Middle Eastern ', 'ethnicity_Others', 'ethnicity_Pasifika', 'ethnicity_South Asian', 'ethnicity_Turkish', 'ethnicity_Unknown', 'ethnicity_White-European', 'relation_Health care professional', 'relation_Others', 'relation_Parent', 'relation_Relative', 'relation_Self', 'relation_Unknown', 'country_grouped_Afghanistan', 'country_grouped_Australia', 'country_grouped_Canada', 'country_grouped_India', 'country_grouped_Jordan', 'country_grouped_New Zealand', 'country_grouped_Other', 'country_grouped_Sri Lanka', 'country_grouped_United Arab Emirates', 'country_grouped_United Kingdom', 'country_grouped_United States']


---------------------------------------------------------------
Step 4: Standardize the continuous column(s).
Binary columns are already 0/1, so they're implicitly on a
comparable scale to each other -- but NOT to raw age (17-64+).
Without this step, age alone would dominate Euclidean distance
in the k-NN graph, effectively drowning out the AQ-10 behavioral
signal that the whole project is built around.
---------------------------------------------------------------

In [7]:
scaler = StandardScaler()
age_scaled = scaler.fit_transform(df_features[continuous_cols])
df_continuous_scaled = pd.DataFrame(
    age_scaled, columns=continuous_cols, index=df_features.index
)
print(f"\nAge before scaling: mean={df_features['age'].mean():.2f}, std={df_features['age'].std():.2f}")
print(f"Age after scaling:  mean={df_continuous_scaled['age'].mean():.2f}, std={df_continuous_scaled['age'].std():.2f}")


Age before scaling: mean=29.18, std=9.69
Age after scaling:  mean=0.00, std=1.00


---------------------------------------------------------------
Step 5: Assemble final feature matrix.
Order: binary (as-is) + scaled continuous + one-hot categorical.
All columns are now numeric and on comparable scales.
---------------------------------------------------------------

In [8]:
df_binary = df_features[binary_cols].astype(float)

X = pd.concat([df_binary, df_continuous_scaled, df_categorical_encoded], axis=1)
X = X.astype(float)

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Total feature dimensions: {X.shape[1]}")
print(f"Feature columns:\n{list(X.columns)}")

# sanity check: no NaNs, no leaked target column
assert X.isnull().sum().sum() == 0, "NaNs found in final feature matrix!"
assert "Class/ASD" not in X.columns, "Target leaked into feature matrix!"


Final feature matrix shape: (704, 42)
Total feature dimensions: 42
Feature columns:
['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'gender', 'jaundice', 'family_autism_history', 'age', 'ethnicity_Asian', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Latino', 'ethnicity_Middle Eastern ', 'ethnicity_Others', 'ethnicity_Pasifika', 'ethnicity_South Asian', 'ethnicity_Turkish', 'ethnicity_Unknown', 'ethnicity_White-European', 'relation_Health care professional', 'relation_Others', 'relation_Parent', 'relation_Relative', 'relation_Self', 'relation_Unknown', 'country_grouped_Afghanistan', 'country_grouped_Australia', 'country_grouped_Canada', 'country_grouped_India', 'country_grouped_Jordan', 'country_grouped_New Zealand', 'country_grouped_Other', 'country_grouped_Sri Lanka', 'country_grouped_United Arab Emirates', 'country_grouped_United Kingdom', 'country_grouped_United States']


---------------------------------------------------------------
Save artifacts for Phase 3 (graph construction)
---------------------------------------------------------------

In [9]:
X.to_csv(OUT_X_PATH, index=False)
y.to_csv(OUT_Y_PATH, index=False)
print(f"\nSaved feature matrix to {OUT_X_PATH}")
print(f"Saved labels to {OUT_Y_PATH}")


Saved feature matrix to Data/X_features.csv
Saved labels to Data/y_labels.csv
